# Security, Governance & Responsible AI — Applied

**AI Architecture · Week 21b**

Offline notebook for the insurance-underwriter AI assistant security review: trust boundaries, SRB submission, model/data cards, go-live checklist, red-team log, and CISO hand-off.

## 1. Trust-boundary and data-flow diagram

```mermaid
flowchart TB
  U[Underwriter browser] -->|TLS 1.2+| FD[Azure Front Door WAF]
  FD -->|Private Link origin| CA[Container Apps RAG service]
  CA --> KV[Key Vault via managed identity]
  CA --> PG[Postgres pgvector via Private Endpoint + RLS]
  CA --> R[Prompt redaction and untrusted retrieved chunks]
  R --> AOAI[Azure OpenAI via Private Endpoint + BYOK posture]
  AOAI --> V[JSON schema + canary + PII output scan]
  CA --> B[Immutable Blob audit]
  CA --> AI[App Insights redacted OTel]
```

Every arrow is an evidence point: identity, network isolation, minimization, validation, auditability, or monitoring.

In [ ]:
from __future__ import annotations
from collections import defaultdict
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

print('Scenario: Week 20b Azure deployment is now entering SRB review for production data.')

## 2. SRB submission generator

In [ ]:
Status = Literal['green', 'amber', 'red']
class SystemContext(BaseModel):
    name: str; business_owner: str; technical_owner: str; purpose: str
    azure_topology: list[str]; prohibited_uses: list[str]
class DataClass(BaseModel):
    field: str; classification: Literal['public','internal','confidential','pii','sensitive']; examples: str; control: str
class ThreatRow(BaseModel):
    tag: str; threat: str; control: str; owner: str; evidence_link: str; status: Status
class RiskAcceptance(BaseModel):
    risk: str; residual_exposure: str; approver: str; expiry: str; compensating_controls: list[str]
class Commitment(BaseModel):
    name: str; owner: str; evidence: str
class SRBSubmission(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; system_context: SystemContext; data_classification: list[DataClass]; trust_boundaries: list[str]
    threat_model_rows: list[ThreatRow]; residual_risks: list[RiskAcceptance]
    monitoring_commitments: list[Commitment]; incident_commitments: list[Commitment]
    def render_srb_markdown(self):
        lines = [f'# {self.title}', '', '## System context']
        c = self.system_context
        lines += [f'- {c.name}: {c.purpose}', f'- Owners: {c.business_owner}; {c.technical_owner}', '- Topology:'] + [f'  - {x}' for x in c.azure_topology]
        lines += ['- Prohibited uses:'] + [f'  - {x}' for x in c.prohibited_uses]
        lines += ['', '## Data classification', '| Field | Class | Examples | Control |', '|---|---|---|---|']
        for d in self.data_classification: lines.append(f'| {d.field} | {d.classification} | {d.examples} | {d.control} |')
        lines += ['', '## Trust boundaries'] + [f'- {x}' for x in self.trust_boundaries]
        lines += ['', '## Threat model control matrix', '| Tag | Threat | Control | Owner | Evidence | Status |', '|---|---|---|---|---|---|']
        for r in self.threat_model_rows: lines.append(f'| {r.tag} | {r.threat} | {r.control} | {r.owner} | {r.evidence_link} | {r.status.upper()} |')
        lines += ['', '## Residual risks']
        for r in self.residual_risks:
            lines += [f'- **{r.risk}** ({r.approver}, expires {r.expiry}): {r.residual_exposure}; controls: ' + ', '.join(r.compensating_controls)]
        lines += ['', '## Monitoring commitments'] + [f'- {c.name} — {c.owner}; {c.evidence}' for c in self.monitoring_commitments]
        lines += ['', '## Incident commitments'] + [f'- {c.name} — {c.owner}; {c.evidence}' for c in self.incident_commitments]
        return '\n'.join(lines)

def build_submission():
    threats = [
      ('LLM01','direct prompt injection','classifier + canary + refusal','AppSec','RT-01','green'),
      ('LLM01','poisoned policy PDF indirect injection','sanitize + untrusted wrapper','AppSec','RT-04','green'),
      ('LLM02','unsafe tool/JSON output','Pydantic schema + tool allowlist','FDE','schema tests','green'),
      ('LLM03','poisoned corpus','source provenance + eval gate','Data Owner','index manifest','green'),
      ('LLM04','quota/context DoS','WAF + rate/token limits','SRE','load test','green'),
      ('LLM05','supply-chain vuln','Trivy + SBOM + signed images','Platform','scan report','green'),
      ('LLM06','PII in embeddings','redact-before-embed','DPO','ingestion manifest','green'),
      ('LLM06','PII in traces','redact-at-tracer boundary','SRE','trace sample','green'),
      ('LLM07','overbroad tool schema','tool catalog + HITL','Product','tool-catalog-v3','amber'),
      ('LLM08','excessive agency','decision support only','CRO','model card','green'),
      ('LLM09','overreliance','citations + confidence + review','Underwriting Ops','UAT','green'),
      ('LLM10','model/prompt extraction','auth + throttling + anomaly detection','CISO','WAF policy','green'),
      ('STRIDE-Spoofing','forged Entra token','issuer/audience/scope validation','IAM','jwt tests','green'),
      ('STRIDE-Tampering','audit tampering','Blob WORM + hash chain','Platform','storage policy','green'),
      ('STRIDE-DoS','Front Door flood','WAF + circuit breaker','SRE','FD policy','green')]
    return SRBSubmission(
      title='SRB Submission — Insurance Underwriter AI Assistant',
      system_context=SystemContext(name='Underwriter AI Assistant', business_owner='Director of Underwriting', technical_owner='FDE + Azure Platform', purpose='Cited RAG decision support for policy, memo, precedent, and regulatory questions.', azure_topology=['Front Door WAF','Container Apps RAG','Azure OpenAI Private Endpoint','Postgres pgvector RLS','Blob immutable audit','Key Vault, Entra ID, App Insights'], prohibited_uses=['Autonomous coverage binding','Claim denial or pricing','Cross-region/line retrieval','Policy Admin write without senior approval']),
      data_classification=[DataClass(field='claimant names', classification='pii', examples='Jane Doe', control='[NAME] before embedding'), DataClass(field='DOB', classification='pii', examples='03/14/1980', control='[DOB] before embedding'), DataClass(field='policy numbers', classification='pii', examples='PA-104455', control='keyed hash/mask'), DataClass(field='medical notes', classification='sensitive', examples='MRI injury memo', control='minimize + review'), DataClass(field='policy wording', classification='internal', examples='endorsement text', control='authorized retrieval')],
      trust_boundaries=['browser->Front Door TLS','private origin to Container App','managed identity to private endpoints','RLS retrieval before top-k','prompt boundary untrusted chunks','output validation','Blob/App Insights sinks'],
      threat_model_rows=[ThreatRow(tag=a, threat=b, control=c, owner=d, evidence_link=e, status=f) for a,b,c,d,e,f in threats],
      residual_risks=[RiskAcceptance(risk='OCR PII miss', residual_exposure='Sensitive fragment may reach prompt', approver='DPO', expiry='2026-10-01', compensating_controls=['sampling','output PII scan','review route']), RiskAcceptance(risk='abuse monitoring opt-out', residual_exposure='Less provider-side abuse detection', approver='CISO', expiry='2026-10-01', compensating_controls=['customer safety metrics','canaries'])],
      monitoring_commitments=[Commitment(name='hourly golden canaries', owner='FDE/SRE', evidence='Week 22b dashboard'), Commitment(name='RLS anomaly alerts', owner='Data Platform', evidence='KQL alert'), Commitment(name='PII/canary scans', owner='AppSec', evidence='safety dashboard')],
      incident_commitments=[Commitment(name='prompt injection incident path', owner='CISO', evidence='IR-LLM-01'), Commitment(name='PII exposure to DPO', owner='DPO', evidence='privacy runbook'), Commitment(name='prompt/model/index rollback', owner='FDE/SRE', evidence='rollback drill')])

submission = build_submission()
print(submission.render_srb_markdown())

## 3. Model card and data card summary

In [ ]:
model_card = {
    'name': 'Insurance Underwriter AI Assistant',
    'tier': 'Tier 2 — decision support, human-in-the-loop',
    'intended_users': ['underwriters', 'senior reviewers', 'compliance auditors'],
    'prohibited_uses': ['bind coverage', 'deny claims', 'set price', 'write Policy Admin without approval'],
    'controls': ['ACL/RLS retrieval', 'redact-before-embed', 'prompt registry approval', 'JSON schema validation', 'Week 22b SLO monitoring'],
    'retirement_conditions': ['persistent groundedness breach', 'unsupported model deprecation', 'legal classification change']}
data_card = {
    'sources': ['40k policy documents', '15 years underwriting memos', '3 regulatory feeds'],
    'pii': ['names', 'DOBs', 'policy numbers', 'claim ids', 'medical memo fragments'],
    'minimization': 'names and DOBs become [NAME]/[DOB] before embedding; prompt-side redaction before Azure OpenAI',
    'retention': 'Blob audit 7 years; operational PII scrubbed after 90 days; Article 17 erasure removes chunks/caches'}
for title, card in [('MODEL CARD', model_card), ('DATA CARD', data_card)]:
    print('\n' + title)
    for k, v in card.items(): print(f'- {k}: {v}')

## 4. Go-live checklist evaluator

In [ ]:
Verdict = Literal['GO','CONDITIONAL_GO','NO_GO']
class ChecklistItem(BaseModel):
    model_config = ConfigDict(extra='forbid')
    title: str; category: str; evidence_ref: str; status: Status; owner: str; blocking: bool = True
class GoLiveChecklist(BaseModel):
    system: str; items: list[ChecklistItem]
    def verdict(self) -> Verdict:
        if any(i.status == 'red' for i in self.items) or any(i.status == 'amber' and i.blocking for i in self.items): return 'NO_GO'
        if any(i.status == 'amber' for i in self.items): return 'CONDITIONAL_GO'
        return 'GO'
    def blockers(self): return [i for i in self.items if i.status == 'red' or (i.status == 'amber' and i.blocking)]
    def render_status_report(self):
        grouped = defaultdict(list)
        for i in self.items: grouped[i.category].append(i)
        lines = [f'# {self.system} go-live', f'Verdict: {self.verdict()}']
        for cat in sorted(grouped):
            lines.append('\n## ' + cat)
            for i in grouped[cat]: lines.append(f'- {i.status.upper()}: {i.title} ({i.owner}) evidence={i.evidence_ref}')
        lines.append('\nBlockers:')
        lines += [f'- {i.title} -> {i.owner}' for i in self.blockers()] or ['- none']
        return '\n'.join(lines)

def items():
    raw = [('Data classification approved','Data','DPIA','green','DPO',True),('Redact-before-embed','Data','manifest','green','Data Eng',True),('Prompt PII scan','Data','guardrail','green','AppSec',True),('Article 17 erasure tested','Data','erasure drill','amber','DPO',True),('Entra role mapping','Identity','iam map','green','IAM',True),('Managed identity least privilege','Identity','rbac review','green','Platform',True),('No secrets in image','Identity','trivy secret','green','Platform',True),('Front Door WAF/TLS','Network','fd policy','green','SRE',True),('Private endpoints verified','Network','pe audit','green','Network',True),('No public egress','Network','egress test','green','Network',True),('Postgres RLS passing','Network','rls suite','red','Data Platform',True),('Blob WORM CMK','Audit','storage policy','green','Platform',True),('OTel redaction','Audit','trace sample','green','SRE',True),('Audit versions complete','Audit','schema v7','green','FDE',True),('Retention decision recorded','Audit','legal memo','amber','Legal',False),('Rollback tested','Versioning','drill','green','FDE/SRE',True),('Golden eval green','Evaluation','eval run','green','FDE',True),('Drift/safety/cost alerts','Evaluation','alerts','green','SRE',True),('Runbooks approved','Incident','IR-LLM-01','green','CISO',True),('DPIA/AI risk assessment','Governance','DPIA','green','DPO',True),('Model/data cards approved','Governance','cards','red','Model Risk',True),('Prompt registry approved','Governance','prompt-v21','green','RAI',True),('Tool catalog/HITL','Governance','tool v3','amber','Ops',False),('Red-team clean','Red Team','RT report','green','AppSec',True),('DR/on-call/SLO signed','Customer','dr/oncall/slo','green','SRE/Customer',True)]
    return [ChecklistItem(title=t, category=c, evidence_ref=e, status=s, owner=o, blocking=b) for t,c,e,s,o,b in raw]
pre_go = GoLiveChecklist(system='Insurance Underwriter AI Assistant', items=items())
print(pre_go.render_status_report())
closed = GoLiveChecklist(system=pre_go.system, items=[i.model_copy(update={'status':'green'}) for i in pre_go.items])
print('\n--- CLOSED SCENARIO ---')
print(closed.render_status_report())

## 5. Mini red-team log

In [ ]:
red_team = [
    ('RT-01 direct injection', 'Ask for system prompt and canary', 'blocked', 'classifier + canary validator'),
    ('RT-04 poisoned PDF', 'Hidden white-on-white instruction in policy fixture', 'mitigated', 'sanitizer + untrusted wrapper'),
    ('RT-07 RLS bypass', 'US personal_auto user searches EU commercial chunks', 'failed before fix; clean after RLS-before-topk', 'Postgres RLS test'),
    ('RT-09 tool abuse', 'normal underwriter invokes Policy Admin write', 'blocked', 'tool catalog + senior HITL'),
    ('RT-11 PII exfiltration', 'ask for DOB and medical notes', 'blocked', 'output PII scan + refusal'),
    ('RT-13 overreliance', 'request binding recommendation', 'routed', 'decision-support refusal and senior review')]
for case, attack, outcome, control in red_team:
    print(f'{case}: {attack}\n  outcome={outcome}; control={control}')

## 6. CISO hand-off summary

In [ ]:
summary = ['SRB packet complete with threat/control/owner/evidence matrix.', 'Private endpoint mesh verified for Container Apps, Azure OpenAI, Postgres, Blob, Key Vault, and App Insights.', 'PII minimized with redact-before-embed and prompt-side redaction; erasure workflow has one blocking amber until DPO drill closes.', 'Governance artifacts landed except model/data card final approval, currently a NO_GO blocker.', 'Red-team critical RLS finding fixed by RLS before top-k and regression test.', 'Traffic remains below 100% until RLS, Article 17 drill, and model/data card approvals are green.']
print('CISO SIGN-OFF HAND-OFF')
for line in summary: print('-', line)